# Conditional and Bayesian Probability

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working in:", os.getcwd())

Working in: /Users/jamesfamodimu/Documents/energy-efficiency


In [2]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

for d in ["figures", "outputs"]:
    Path(d).mkdir(exist_ok=True)
pd.set_option("display.width", 110); pd.set_option("display.max_columns", 20)
print("Setup is complete")

Setup is complete


In [10]:
from src.data import load_data, SEED, TARGETS, THRESH

random_number_generator = np.random.default_rng(SEED)
df = load_data()

print(data_frame.shape, "| THRESH =", THRESH)

(768, 10) | THRESH = 30


In [15]:
df["high"] = (df["heating_load"] > THRESH).astype(int)
tall_buildings_subset = df[df["height"] == 7.0]

print("P(High)               =", round(df["high"].mean(), 3))
print("P(High | height=3.5)  =", round(df[df["height"] == 3.5]["high"].mean(), 3))
print("P(High | height=7.0)  =", round(tall_buildings_subset["high"].mean(), 3))
print(tall_buildings_subset.groupby("glazing_area")["high"].mean().round(3))

prior_probability = (df["glazing_area"] == 0.40).mean()
likelihood = df[df["glazing_area"] == 0.40]["high"].mean()

bayes_posterior_probability = (likelihood * prior_probability) / df["high"].mean()
direct_count_probability = (df[df["high"] == 1]["glazing_area"] == 0.40).mean()

print("Bayes P(G=0.40 | High) =", round(bayes_posterior_probability, 3), " direct count:", round(direct_count_probability, 3))


P(High)               = 0.262
P(High | height=3.5)  = 0.0
P(High | height=7.0)  = 0.523
glazing_area
0.00    0.000
0.10    0.333
0.25    0.508
0.40    0.833
Name: high, dtype: float64
Bayes P(G=0.40 | High) = 0.498  direct count: 0.498


In [16]:
chi_square_records = []

for factor in ["height", "glazing_area", "orientation", "glazing_dist"]:
    contingency_table = pd.crosstab(df[factor], df["high"])
    chi2_statistic, p_value, degrees_of_freedom, _ = stats.chi2_contingency(contingency_table)
    
    total_observations = contingency_table.values.sum()
    minimum_dimension = min(contingency_table.shape)
    cramers_v_statistic = np.sqrt(chi2_statistic / (total_observations * (minimum_dimension - 1)))
    
    chi_square_records.append([factor, chi2_statistic, degrees_of_freedom, p_value, cramers_v_statistic])

chi_square_table = pd.DataFrame(chi_square_records, columns=["factor", "chi2", "dof", "p", "cramers_V"])
chi_square_table.to_csv("outputs/chi_square.csv", index=False)

print(chi_square_table.round({"chi2": 2, "p": 3, "cramers_V": 3}))


         factor    chi2  dof      p  cramers_V
0        height  269.55    1  0.000      0.592
1  glazing_area   58.13    3  0.000      0.275
2   orientation    0.02    3  0.999      0.005
3  glazing_dist   18.18    5  0.003      0.154


In [18]:
bayesian_estimation_records = []
posterior_random_draws = {}

for glazing_ratio in [0.10, 0.25, 0.40]:
    glazing_subset = tall_buildings_subset[tall_buildings_subset["glazing_area"] == glazing_ratio]
    success_count = glazing_subset["high"].sum()
    total_count = len(glazing_subset)
    
    beta_posterior_distribution = stats.beta(1 + success_count, 1 + total_count - success_count)
    posterior_random_draws[glazing_ratio] = beta_posterior_distribution.rvs(100_000, random_state=SEED)
    
    bayesian_estimation_records.append([
        glazing_ratio, 
        f"{success_count}/{total_count}", 
        beta_posterior_distribution.mean(), 
        beta_posterior_distribution.ppf(0.025), 
        beta_posterior_distribution.ppf(0.975)
    ])

probability_greater = (posterior_random_draws[0.40] > posterior_random_draws[0.10]).mean()
print("P(theta_0.40 > theta_0.10) =", probability_greater)

table_columns = ["glazing", "k/n", "post_mean", "cred_2.5%", "cred_97.5%"]
bayesian_results_table = pd.DataFrame(bayesian_estimation_records, columns=table_columns)
bayesian_results_table.to_csv("outputs/bayes_beta.csv", index=False)

print(bayesian_results_table.round(3))


P(theta_0.40 > theta_0.10) = 1.0
   glazing      k/n  post_mean  cred_2.5%  cred_97.5%
0     0.10   40/120      0.336      0.255       0.422
1     0.25   61/120      0.508      0.420       0.596
2     0.40  100/120      0.828      0.756       0.889


# Estimation and Hypothesis Testing

In [19]:
def calculate_bootstrap_confidence_interval(data_array, bootstrap_iterations=10_000):
    bootstrap_samples = random_number_generator.choice(
        data_array, 
        size=(bootstrap_iterations, len(data_array)), 
        replace=True
    )
    bootstrap_means = bootstrap_samples.mean(axis=1)
    return np.percentile(bootstrap_means, [2.5, 97.5])

confidence_interval_records = []

grouped_data_sources = [
    ("all", data_frame["heating_load"]),
    ("3.5 m", data_frame[data_frame["height"] == 3.5]["heating_load"]),
    ("7.0 m", data_frame[data_frame["height"] == 7.0]["heating_load"])
]

for group_label, sample_data in grouped_data_sources:
    degrees_of_freedom = len(sample_data) - 1
    sample_mean = sample_data.mean()
    standard_error = stats.sem(sample_data)
    
    t_distribution_interval = stats.t.interval(
        confidence=0.95, 
        df=degrees_of_freedom, 
        loc=sample_mean, 
        scale=standard_error
    )
    
    bootstrap_interval = calculate_bootstrap_confidence_interval(sample_data.values)
    
    confidence_interval_records.append([
        group_label, 
        len(sample_data), 
        sample_mean, 
        *t_distribution_interval, 
        *bootstrap_interval
    ])

table_columns = ["group", "n", "mean", "t_low", "t_high", "boot_low", "boot_high"]
confidence_intervals_table = pd.DataFrame(confidence_interval_records, columns=table_columns)
confidence_intervals_table.to_csv("outputs/confidence_intervals.csv", index=False)

print(confidence_intervals_table.round(2))


   group    n   mean  t_low  t_high  boot_low  boot_high
0    all  768  22.31  21.59   23.02     21.59      23.02
1  3.5 m  384  13.34  13.07   13.61     13.07      13.60
2  7.0 m  384  31.28  30.68   31.87     30.69      31.87


In [20]:
heating_load_series = df["heating_load"]
low_height_load = heating_load_series[df["height"] == 3.5]
high_height_load = heating_load_series[df["height"] == 7.0]
hypothesis_test_records = []

welch_t_test = stats.ttest_ind(high_height_load, low_height_load, equal_var=False)
cohens_d_effect = (high_height_load.mean() - low_height_load.mean()) / np.sqrt((high_height_load.var() + low_height_load.var()) / 2)
hypothesis_test_records.append([
    "H1 height (Welch t)", 
    welch_t_test.statistic, 
    welch_t_test.pvalue, 
    f"d = {cohens_d_effect:.2f}"
])

for factor_name, hypothesis_label in [("orientation", "H2"), ("glazing_area", "H3")]:
    anova_linear_model = smf.ols(f"heating_load ~ C({factor_name})", data=df).fit()
    anova_table = sm.stats.anova_lm(anova_linear_model)
    eta_squared = anova_table["sum_sq"].iloc[0] / anova_table["sum_sq"].sum()
    hypothesis_test_records.append([
        f"{hypothesis_label} {factor_name} (ANOVA)", 
        anova_table["F"].iloc[0], 
        anova_table["PR(>F)"].iloc[0], 
        f"eta2 = {eta_squared:.3f}"
    ])

two_way_anova_model = smf.ols("heating_load ~ C(height) * C(glazing_area)", data=df).fit()
two_way_anova_table = sm.stats.anova_lm(two_way_anova_model, typ=2)
interaction_row = two_way_anova_table.loc["C(height):C(glazing_area)"]
hypothesis_test_records.append([
    "H4 height x glazing (2-way ANOVA)", 
    interaction_row["F"], 
    interaction_row["PR(>F)"], 
    "interaction"
])

paired_t_test = stats.ttest_rel(df["cooling_load"], heating_load_series)
mean_difference = (df["cooling_load"] - heating_load_series).mean()
hypothesis_test_records.append([
    "H5 cooling vs heating (paired t)", 
    paired_t_test.statistic, 
    paired_t_test.pvalue,
    f"mean diff = {mean_difference:.2f}"
])

hypothesis_tests_table = pd.DataFrame(
    hypothesis_test_records, 
    columns=["test", "statistic", "p", "effect"]
)
hypothesis_tests_table["p_bonf"] = multipletests(hypothesis_tests_table["p"], method="bonferroni")[1]
hypothesis_tests_table["reject_H0"] = hypothesis_tests_table["p_bonf"] < 0.05

hypothesis_tests_table.to_csv("outputs/hypothesis_tests.csv", index=False)

display_table = hypothesis_tests_table.copy()
display_table["statistic"] = display_table["statistic"].round(2)

for column_name in ["p", "p_bonf"]:
    display_table[column_name] = display_table[column_name].apply(
        lambda value: "< 0.001" if value < 0.001 else f"{value:.3f}"
    )

print(display_table)


                                test  statistic        p            effect   p_bonf  reject_H0
0                H1 height (Welch t)      53.86  < 0.001          d = 3.89  < 0.001       True
1             H2 orientation (ANOVA)       0.01    0.999      eta2 = 0.000    1.000      False
2            H3 glazing_area (ANOVA)      22.59  < 0.001      eta2 = 0.081  < 0.001       True
3  H4 height x glazing (2-way ANOVA)      14.09  < 0.001       interaction  < 0.001       True
4   H5 cooling vs heating (paired t)      28.36  < 0.001  mean diff = 2.28  < 0.001       True
